# Aumentando el realismo en modelos de AgentPy: Flocking Behavior

TC2008B · Tec de Monterrey

**Integrantes:** Carlos Eduardo Arias Capetillo · Ilan David Narváez Martínez · Joel Martínez Escalera · Mario Giovanni González López · Oswaldo García Torres

**Cómo ejecutar:** `pip install -r requirements.txt` y luego *Run All*. Todas las corridas usan una semilla fija (ver sección 0).

**Índice**
0. Configuración
1. Parte I: modelo base y preguntas guía
2. Parte II: mejoras candidatas y priorización
3. Parte III: modelo extendido
4. Calibración
5. Recolección de datos (KPIs)
6. Experimentos
7. Validación
8. Conclusiones y limitaciones
9. Referencias

## 0. Configuración

In [ ]:
# Model design
import agentpy as ap
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import IPython

## 1. Parte I: modelo base y preguntas guía


### 1.0 Modelo base (sin modificar)

**Sobre el modelo.** Es el modelo de *boids* de Craig Reynolds (1986): cada agente ajusta su movimiento con tres reglas locales basadas en sus vecinos cercanos.
- **Separación:** alejarse de vecinos demasiado cercanos.
- **Alineación:** orientarse hacia la dirección promedio de los vecinos.
- **Cohesión:** moverse hacia la posición promedio de los vecinos.

La implementación de AgentPy sigue el *Boids Pseudocode* de Conrad Parker y añade una cuarta regla para evitar los bordes del espacio. Funciona en 2 o 3 dimensiones.

**Estructura.** Hay dos clases: una para los agentes (`Boid`) y otra para el modelo (`BoidsModel`). Cada agente empieza con posición y velocidad aleatorias (arreglos de numpy). La posición vive en el espacio, al que el agente accede con `pos` y `neighbors()`. `update_velocity()` y `update_position()` están separados para que **todos** los agentes calculen su nueva velocidad antes de que cualquiera se mueva.

#### Función auxiliar

In [ ]:
def normalize(v):
    """ Normalize a vector to length 1. """
    norm = np.linalg.norm(v)
    if norm == 0:
        return v
    return v / norm

#### Agente

In [ ]:
class Boid(ap.Agent):
    """ An agent with a position and velocity in a continuous space,
    who follows Craig Reynolds three rules of flocking behavior;
    plus a fourth rule to avoid the edges of the simulation space. """

    def setup(self):

        self.velocity = normalize(
            self.model.nprandom.random(self.p.ndim) - 0.5)

    def setup_pos(self, space):

        self.space = space
        self.neighbors = space.neighbors
        self.pos = space.positions[self]

    def update_velocity(self):

        pos = self.pos
        ndim = self.p.ndim

        # Rule 1 - Cohesion
        nbs = self.neighbors(self, distance=self.p.outer_radius)
        nbs_len = len(nbs)
        nbs_pos_array = np.array(nbs.pos)
        nbs_vec_array = np.array(nbs.velocity)
        if nbs_len > 0:
            center = np.sum(nbs_pos_array, 0) / nbs_len
            v1 = (center - pos) * self.p.cohesion_strength
        else:
            v1 = np.zeros(ndim)

        # Rule 2 - Seperation
        v2 = np.zeros(ndim)
        for nb in self.neighbors(self, distance=self.p.inner_radius):
            v2 -= nb.pos - pos
        v2 *= self.p.seperation_strength

        # Rule 3 - Alignment
        if nbs_len > 0:
            average_v = np.sum(nbs_vec_array, 0) / nbs_len
            v3 = (average_v - self.velocity) * self.p.alignment_strength
        else:
            v3 = np.zeros(ndim)

        # Rule 4 - Borders
        v4 = np.zeros(ndim)
        d = self.p.border_distance
        s = self.p.border_strength
        for i in range(ndim):
            if pos[i] < d:
                v4[i] += s
            elif pos[i] > self.space.shape[i] - d:
                v4[i] -= s

        # Update velocity
        self.velocity += v1 + v2 + v3 + v4
        self.velocity = normalize(self.velocity)

    def update_position(self):

        self.space.move_by(self, self.velocity)

#### Modelo

In [ ]:
class BoidsModel(ap.Model):
    """
    An agent-based model of animals' flocking behavior,
    based on Craig Reynolds' Boids Model [1]
    and Conrad Parkers' Boids Pseudocode [2].

    [1] http://www.red3d.com/cwr/boids/
    [2] http://www.vergenet.net/~conrad/boids/pseudocode.html
    """

    def setup(self):
        """ Initializes the agents and network of the model. """

        self.space = ap.Space(self, shape=[self.p.size]*self.p.ndim)
        self.agents = ap.AgentList(self, self.p.population, Boid)
        self.space.add_agents(self.agents, random=True)
        self.agents.setup_pos(self.space)

    def step(self):
        """ Defines the models' events per simulation step. """

        self.agents.update_velocity()   # Adjust direction
        self.agents.update_position()   # Move into new direction

#### Visualización

`animation_plot` recibe el modelo y los parámetros, y crea una animación con `ap.animate()`.

In [ ]:
def animation_plot_single(m, ax):
    ndim = m.p.ndim
    ax.set_title(f"Boids Flocking Model {ndim}D t={m.t}")
    pos = m.space.positions.values()
    pos = np.array(list(pos)).T  # Transform
    ax.scatter(*pos, s=1, c='black')
    ax.set_xlim(0, m.p.size)
    ax.set_ylim(0, m.p.size)
    if ndim == 3:
        ax.set_zlim(0, m.p.size)
    ax.set_axis_off()

def animation_plot(m, p):
    projection = '3d' if p['ndim'] == 3 else None
    fig = plt.figure(figsize=(7,7))
    ax = fig.add_subplot(111, projection=projection)
    animation = ap.animate(m(p), fig, ax, animation_plot_single)
    return IPython.display.HTML(animation.to_jshtml(fps=20))

#### Simulación 2D

Los parámetros se definen en un diccionario.

In [ ]:
parameters2D = {
    'size': 50,
    'seed': 123,
    'steps': 200,
    'ndim': 2,
    'population': 200,
    'inner_radius': 3,
    'outer_radius': 10,
    'border_distance': 10,
    'cohesion_strength': 0.005,
    'seperation_strength': 0.1,
    'alignment_strength': 0.3,
    'border_strength': 0.5
}

In [ ]:
animation_plot(BoidsModel, parameters2D)

#### Simulación 3D

Lo mismo en tres dimensiones, con más agentes.

In [ ]:
new_parameters = {
    'ndim': 3,
    'population': 1000,
}

parameters3D = dict(parameters2D)
parameters3D.update(new_parameters)

animation_plot(BoidsModel, parameters3D)

#### Métricas de apoyo

Dos funciones para medir el estado de la bandada. Se usan en las respuestas de abajo y se reutilizarán como KPIs en la sección 5.
- **Polarización:** magnitud de la velocidad promedio. Como cada velocidad mide 1, vale 0 si cada agente va en una dirección distinta y 1 si todos van en la misma.
- **Distancia al vecino más cercano:** promedio, sobre todos los agentes, de la distancia a su vecino más próximo. Mide qué tan compacta está la bandada.

In [ ]:
def polarizacion(m):
    """ Magnitud de la velocidad promedio de los agentes (0 = desorden, 1 = alineación total). """
    return np.linalg.norm(np.mean([a.velocity for a in m.agents], axis=0))

def dist_vecino_mas_cercano(m):
    """ Distancia promedio de cada agente a su vecino más cercano. """
    pos = np.array(list(m.space.positions.values()))
    d = np.linalg.norm(pos[:, None] - pos[None], axis=-1)
    np.fill_diagonal(d, np.inf)
    return d.min(axis=1).mean()

### 1. Estructura del modelo

**(a) Componentes principales de un modelo AgentPy**

*Respuesta:* Un modelo de AgentPy se compone de:
- **Parámetros:** un diccionario (`parameters2D`) al que el modelo y los agentes acceden como `self.p`.
- **Modelo:** una subclase de `ap.Model` (`BoidsModel`) que define el ciclo de la simulación.
- **Agentes:** una subclase de `ap.Agent` (`Boid`), agrupados en una `ap.AgentList`.
- **Entorno:** aquí un `ap.Space` continuo.
- **Generadores aleatorios:** `model.random` (Python) y `model.nprandom` (numpy), sembrados con el parámetro `seed`.
- **Salida:** un `ap.DataDict` en `model.output` con la información, los parámetros y los datos registrados.

**(b) Cómo se definen y relacionan `ap.Model` y `ap.Agent`**

*Respuesta:* `BoidsModel` hereda de `ap.Model` y `Boid` de `ap.Agent`. El modelo **contiene** a los agentes y al espacio: en su `setup()` crea `self.space` y `self.agents = ap.AgentList(self, population, Boid)`. Cada agente recibe al crearse una referencia a su modelo (`self.model`), y a través de ella accede a los parámetros (`self.p`) y a los generadores aleatorios (`self.model.nprandom`). El modelo controla el tiempo y llama a los métodos de los agentes; los agentes no se llaman entre sí, solo leen el estado de sus vecinos a través del espacio.

**(c) Diferencia entre `setup()` y `step()`**

*Respuesta:* `setup()` corre **una sola vez**, en t = 0, y prepara la simulación: crea el espacio, los agentes y sus posiciones. `step()` corre **en cada paso** (t = 1, 2, …, `steps`) y define la dinámica: primero todos actualizan su velocidad y luego todos se mueven. El ciclo completo de `model.run()` es `setup → update → [t += 1 → step → update] × steps → end`. Además, `Boid` tiene su propio `setup()`, que AgentPy llama al crear cada agente y que le asigna una velocidad inicial.

### 2. Agentes

**(a) Cómo se crean y reciben atributos**

*Respuesta:* En tres pasos dentro de `BoidsModel.setup()`:
1. `ap.AgentList(self, self.p.population, Boid)` crea 200 agentes. Al crear cada uno se ejecuta `Boid.setup()`, que le asigna una velocidad aleatoria de magnitud 1.
2. `self.space.add_agents(self.agents, random=True)` les asigna una posición aleatoria en el espacio.
3. `self.agents.setup_pos(self.space)` llama ese método en cada agente para que guarde referencias al espacio, a la función de vecinos y a su posición.

Todos los agentes leen los mismos parámetros (`self.p`), así que no hay atributos individuales: solo difieren en posición y velocidad iniciales.

**(b) Variables o estados de cada agente**

*Respuesta:*
- `velocity`: vector de dimensión `ndim` y magnitud 1 (dirección de movimiento).
- `pos`: posición. No es una copia: es **el mismo arreglo** que guarda el espacio, así que se actualiza solo cuando el agente se mueve (verificado abajo).
- `space` y `neighbors`: referencias al espacio y a su función de búsqueda de vecinos.
- `id`: identificador que asigna AgentPy.

No tienen otro estado interno (energía, rol, memoria, objetivo).

**(c) Cómo perciben su entorno**

*Respuesta:* Con `self.neighbors(self, distance=r)`, que usa un árbol k-d para devolver **todos** los agentes dentro de un radio euclidiano `r`. Hay dos radios: `outer_radius = 10` para cohesión y alineación, e `inner_radius = 3` para separación. La percepción es de 360°, sin límite de vecinos y perfecta: cada agente conoce la posición y la velocidad exactas de sus vecinos, sin ruido. Además, cada agente "percibe" los bordes comparando su propia posición con el tamaño del espacio.

In [ ]:
m = BoidsModel(parameters2D)
m.sim_setup()  # Solo ejecuta setup(), sin avanzar pasos
a = m.agents[0]
print('Atributos propios del agente:', [k for k in vars(a) if not k.startswith('_')])
print('velocity =', a.velocity, '| magnitud =', np.linalg.norm(a.velocity))
print('pos =', a.pos)
print('¿pos es el mismo arreglo que guarda el espacio?', a.pos is m.space.positions[a])
print('Vecinos en outer_radius:', len(a.neighbors(a, distance=m.p.outer_radius)),
      '| en inner_radius:', len(a.neighbors(a, distance=m.p.inner_radius)))

### 3. Ambiente

**(a) Tipo de entorno**

*Respuesta:* Un `ap.Space`, es decir, un espacio **continuo** (las posiciones son números reales, no celdas). Su forma es `[size] * ndim`: un cuadrado de 50 × 50 en 2D o un cubo de 50 × 50 × 50 en 3D. No es toroidal (`torus=False` por defecto), así que los bordes son paredes y no se conectan entre sí.

**(b) Cómo se colocan los agentes**

*Respuesta:* `add_agents(..., random=True)` les da a cada uno una posición uniforme en [0, size) en cada eje, usando `model.random`. No hay agrupamiento inicial ni control de densidad: con 200 agentes en 2500 unidades², la densidad es de 0.08 agentes por unidad².

**(c) Reglas espaciales que afectan movimiento e interacción**

*Respuesta:*
- **Vecindad métrica:** solo interactúan los agentes dentro de un radio, sin importar cuántos sean.
- **Bordes suaves (regla 4):** si un agente está a menos de `border_distance = 10` de un borde, se suma `border_strength = 0.5` hacia adentro en ese eje.
- **Bordes duros:** si aun así un movimiento lo sacaría del espacio, AgentPy recorta la posición al límite (`[0, size]`).
- No hay obstáculos ni zonas con propiedades distintas.

Con `size = 50` y `border_distance = 10`, la zona libre de la regla de bordes es solo el cuadrado central de 30 × 30, así que **el 64% del área está bajo influencia de los bordes**. El borde es un factor dominante en este modelo.

In [ ]:
p = parameters2D
libre = (p['size'] - 2 * p['border_distance']) ** p['ndim']
print(f"Fracción del espacio bajo influencia de bordes: {1 - libre / p['size'] ** p['ndim']:.0%}")
print('Densidad inicial (agentes por unidad de área):', p['population'] / p['size'] ** p['ndim'])

### 4. Reglas de comportamiento

**(a) Cómo se codifica el comportamiento en `step()`**

*Respuesta:* `BoidsModel.step()` hace dos llamadas: `self.agents.update_velocity()` y después `self.agents.update_position()`. En `update_velocity()`, cada agente calcula cuatro vectores:
- **v1, cohesión:** `(centro de los vecinos − pos) × cohesion_strength`, con los vecinos en `outer_radius`.
- **v2, separación:** `−Σ (pos del vecino − pos) × seperation_strength`, con los vecinos en `inner_radius`.
- **v3, alineación:** `(velocidad promedio de los vecinos − velocidad propia) × alignment_strength`.
- **v4, bordes:** empuje de `border_strength` hacia adentro si está cerca de un borde.

Suma los cuatro a su velocidad y la **normaliza a magnitud 1**. Después, `update_position()` mueve al agente `velocity` unidades con `space.move_by`.

**(b) Reglas de decisión, procesos aleatorios o condicionales**

*Respuesta:* Solo hay condicionales simples: si no hay vecinos, la cohesión y la alineación valen cero; y la regla de bordes revisa eje por eje si el agente está cerca de una pared. No hay decisiones discretas, estados ni roles, y **la dinámica no tiene azar**: después de la inicialización, todo es determinista.

**(c) Cómo interactúan entre sí y con el ambiente**

*Respuesta:* Entre agentes, la interacción es **indirecta y local**: cada uno lee las posiciones y velocidades de sus vecinos y ajusta su propia dirección; no hay comunicación ni contacto. Con el ambiente, la única interacción son los bordes.

Dos observaciones:
- Como la velocidad se normaliza, **todos se mueven siempre a rapidez 1**; las reglas solo cambian la dirección.
- La separación es **proporcional a la distancia**: un vecino a 2.9 unidades empuja más que uno a 0.5. Es lo contrario de lo que se esperaría físicamente; otras implementaciones de boids usan una fuerza que crece al acercarse (por ejemplo, 1/d).

### 5. Agendamiento

**(a) Qué scheduler se usa**

*Respuesta:* AgentPy no tiene clases `RandomActivation` ni `SequentialActivation` (esas son de la librería Mesa). Aquí, llamar un método sobre una `AgentList` (`self.agents.update_velocity()`) lo ejecuta en cada agente **en el orden de la lista**, que es el orden de creación y siempre es el mismo. En términos de la pregunta, equivale a **activación secuencial fija**. El paso está dividido en dos fases: todos calculan su velocidad y, después, todos se mueven.

**(b) Cómo afecta el orden de activación a los resultados**

*Respuesta:*
- **En las posiciones, no afecta:** nadie se mueve hasta que todos calcularon su velocidad, así que cada agente ve las posiciones del mismo instante.
- **En las velocidades, sí afecta:** `velocity` se actualiza en el momento en que cada agente la calcula. Por eso, en la regla de alineación, un agente que se activa tarde ya ve las velocidades **nuevas** de los que se activaron antes, mientras que uno que se activa temprano ve las viejas.

En la prueba de abajo, con la misma semilla, el orden fijo y un orden barajado en cada paso terminan con posiciones distintas. Sin embargo, la polarización final promedio sobre 5 semillas es parecida (≈ 0.84 vs ≈ 0.86). Es decir, el orden cambia las trayectorias individuales, pero con 5 semillas no se nota en el comportamiento colectivo; habría que confirmarlo con ≥ 20 réplicas. Una actualización realmente sincrónica guardaría primero una copia de todas las velocidades.

In [ ]:
class BoidsModelOrdenAleatorio(BoidsModel):
    """ Igual que el modelo base, pero baraja el orden de activación en cada paso. """
    def step(self):
        self.agents.shuffle().update_velocity()
        self.agents.update_position()

def posiciones_finales(M, p):
    m = M(p)
    m.run(display=False)
    return np.array(list(m.space.positions.values())), polarizacion(m)

pos_fijo, _ = posiciones_finales(BoidsModel, parameters2D)
pos_aleatorio, _ = posiciones_finales(BoidsModelOrdenAleatorio, parameters2D)
print('Misma semilla, orden fijo vs barajado → posiciones idénticas:', np.array_equal(pos_fijo, pos_aleatorio))

for nombre, M in [('Orden fijo (base)', BoidsModel), ('Orden barajado', BoidsModelOrdenAleatorio)]:
    pols = [posiciones_finales(M, dict(parameters2D, seed=s))[1] for s in range(5)]
    print(f'{nombre}: polarización final en 5 semillas = {np.round(pols, 3)}, media = {np.mean(pols):.3f}')

### 6. Aleatoriedad y reproducibilidad

**(a) Cómo se introduce aleatoriedad**

*Respuesta:* Solo en las **condiciones iniciales**:
- **Posición:** uniforme en el espacio, con `model.random` (generador de Python).
- **Velocidad:** `normalize(nprandom.random(ndim) − 0.5)`, con `model.nprandom` (generador de numpy). Toma un punto uniforme en un cuadrado y lo normaliza, así que la dirección no es exactamente uniforme: favorece un poco las diagonales.

Después de t = 0 no hay más azar; la dinámica es determinista.

**(b) Cómo reproducir resultados con semilla fija**

*Respuesta:* Con el parámetro `'seed': 123`. Al iniciar, AgentPy crea `model.random` con esa semilla y siembra `model.nprandom` con un número derivado de él, así que una sola semilla controla ambos generadores. La prueba de abajo confirma que la misma semilla da exactamente las mismas posiciones finales y una semilla distinta no. Para réplicas, `ap.Experiment(..., randomize=True)` genera una semilla distinta por réplica a partir de la del `Sample`.

Para que esto se mantenga en las mejoras, **todo el azar debe venir de `self.model.random` o `self.model.nprandom`**; usar `np.random` o `random` directamente rompería la reproducibilidad.

In [ ]:
def posiciones(seed):
    m = BoidsModel(dict(parameters2D, seed=seed))
    m.run(display=False)
    return np.array(list(m.space.positions.values()))

print('Misma semilla (123 y 123) → resultados idénticos:', np.array_equal(posiciones(123), posiciones(123)))
print('Semilla distinta (123 y 7) → resultados idénticos:', np.array_equal(posiciones(123), posiciones(7)))

### 7. Colección de datos

**(a) Cómo recolecta datos el modelo**

*Respuesta:* El modelo base **no recolecta nada**: no llama a `record()` ni a `report()`, y no define `update()` ni `end()`. Su salida solo contiene información de la corrida, los parámetros y la semilla (ver abajo). AgentPy ofrece:
- `self.record('nombre', valor)` dentro de `update()`, para guardar una variable en cada paso. También funciona en agentes, por ejemplo `self.agents.record('velocity')`.
- `self.report('nombre', valor)` dentro de `end()`, para guardar un valor final de la corrida.

`model.run()` devuelve un `ap.DataDict` con `variables` (series de tiempo) y `reporters` (valores finales) como DataFrames de pandas.

**(b) Variables clave rastreadas en el tiempo**

*Respuesta:* Ninguna en el modelo base. Las que proponemos, porque miden directamente lo que producen las reglas:
- **Polarización:** efecto de la alineación.
- **Distancia al vecino más cercano:** equilibrio entre cohesión y separación.
- Candidatas adicionales: dispersión del grupo, número de grupos, fracción de agentes en la zona de bordes.

En la celda de abajo se registran las dos primeras con `record()`. La polarización sube de ~0.1 a ~0.9 y la distancia al vecino baja de ~1.8 a ~1.05.

**(c) Cómo se usan los datos para análisis o visualización**

*Respuesta:* En el ejemplo, solo para la animación: `ap.animate` lee `m.space.positions` en cada paso y dibuja los puntos. No hay análisis cuantitativo. Con `record()`, el `DataDict` se convierte en DataFrames para graficar series de tiempo, comparar configuraciones y calcular intervalos de confianza entre réplicas.

In [ ]:
m = BoidsModel(parameters2D)
salida = m.run(display=False)
print('Lo que guarda el modelo base:', list(salida.keys()))
print(salida.reporters)

class BoidsModelConDatos(BoidsModel):
    """ Modelo base + registro de dos métricas en cada paso. """
    def update(self):
        self.record('polarizacion', polarizacion(self))
        self.record('dist_vecino', dist_vecino_mas_cercano(self))

datos = BoidsModelConDatos(parameters2D).run(display=False)
serie = datos.variables.BoidsModelConDatos
ax = serie.plot(subplots=True, figsize=(7, 4), title='Modelo base, una corrida (seed = 123)')
plt.xlabel('Paso t')
plt.tight_layout()
plt.show()
print(serie.loc[[0, 50, 100, 200]].round(3))

### 8. Experimentos

**(a) Cómo ejecutar el modelo varias veces con parámetros distintos**

*Respuesta:* Hay dos formas:
- **Manual:** un ciclo que crea `BoidsModel(p).run()` para cada combinación de parámetros y semilla, y junta los resultados.
- **Con AgentPy:** se define el espacio de parámetros con `ap.Sample`, usando `ap.Values` (lista de valores), `ap.Range` o `ap.IntRange` (rangos), y se ejecuta con `ap.Experiment`.

**(b) Cómo ayuda `ap.Experiment` con barridos o réplicas**

*Respuesta:* `ap.Experiment(Modelo, sample, iterations=n, record=True, randomize=True)` corre cada combinación del `Sample` `n` veces con semillas distintas. Junta todo en un solo `DataDict`, identificando cada corrida por `sample_id` e `iteration`, y puede ejecutarse en paralelo con `n_jobs`. Para 20 réplicas por configuración basta con `iterations=20`.

La celda de abajo es una prueba pequeña que confirma que funciona con las versiones instaladas: dos valores de `alignment_strength`, 3 réplicas y 30 pasos. La polarización final media es mayor con más alineación (~0.27 vs ~0.11).

In [ ]:
class BoidsModelReporta(BoidsModel):
    def end(self):
        self.report('polarizacion_final', polarizacion(self))

muestra = ap.Sample(dict(parameters2D, steps=30, alignment_strength=ap.Values(0.1, 0.3)))
exp = ap.Experiment(BoidsModelReporta, muestra, iterations=3, randomize=True)
resultados = exp.run(display=False)
resultados.arrange_reporters().groupby('alignment_strength')['polarizacion_final'].agg(['mean', 'std', 'count'])

### 9. Validación e interpretación

**(a) ¿Los resultados hacen sentido frente a lo esperado?**

*Respuesta:* Sí, cualitativamente. Se espera que agentes que empiezan desordenados formen grupos alineados y compactos, y eso pasa: la polarización sube de ~0.1 a ~0.9 en 200 pasos y la distancia al vecino más cercano baja de ~1.8 a ~1.05 (sección 7). Dos matices:
- **La evolución no es monótona:** la polarización llega a ~0.79 en t = 50, baja a ~0.67 en t = 100 y sube otra vez. Probablemente son grupos que chocan con los bordes o entre sí y se reorganizan; conviene comprobarlo en la animación.
- **Hay bastante variabilidad entre semillas:** la polarización final va de ~0.69 a ~0.91 en 5 semillas (sección 5). Por eso hacen falta muchas réplicas.

**(b) Cómo verificar que el código hace lo previsto**

*Respuesta:*
- **Reproducibilidad:** la misma semilla da el mismo resultado (sección 6).
- **Invariantes:** en todo momento, cada velocidad mide 1 y cada posición está dentro del espacio.
- **Casos límite:**
  - Un solo agente no tiene vecinos y solo lo afecta la regla de bordes.
  - Con cohesión, separación y alineación en cero, no hay interacción y la polarización se queda baja.
  - Con alineación alta (`alignment_strength = 1`) se esperaría una polarización cercana a 1, pero la prueba da ~0.84, **menos** que con el valor base de 0.3 (~0.90). Es solo una semilla, pero sugiere que más alineación no siempre produce más orden: vale la pena revisarlo en el barrido de parámetros.
- **Revisión visual** con la animación.

La celda de abajo prueba los invariantes y los casos límite.

**(c) Limitaciones del modelo base**

*Respuesta:*
1. **Agentes idénticos:** mismos parámetros, sin roles ni diferencias individuales.
2. **Rapidez constante:** todos van a 1 unidad por paso; las reglas solo cambian la dirección.
3. **Percepción irreal:** 360°, perfecta, sin ruido y con todos los vecinos dentro de un radio fijo. Las aves reales interactúan con un número fijo de vecinos cercanos, unos 6 o 7 (Ballerini et al., 2008).
4. **Dinámica sin ruido:** el único azar está en las condiciones iniciales.
5. **Separación proporcional a la distancia**, en lugar de crecer al acercarse.
6. **Actualización secuencial** con una mezcla de velocidades viejas y nuevas (sección 5).
7. **Entorno vacío y cerrado:** el 64% del área está bajo influencia de los bordes, sin obstáculos, depredadores, recursos ni objetivos.
8. **Sin escala física:** las unidades de espacio y tiempo no corresponden a metros ni segundos.
9. **No registra datos:** solo se puede analizar visualmente.

In [ ]:
# Invariantes durante toda una corrida
m = BoidsModel(parameters2D)
m.sim_setup()
while m.running:
    m.sim_step()
    rapideces = np.linalg.norm([a.velocity for a in m.agents], axis=1)
    pos = np.array(list(m.space.positions.values()))
    assert np.allclose(rapideces, 1), f'Rapidez distinta de 1 en t={m.t}'
    assert (pos >= 0).all() and (pos <= m.p.size).all(), f'Agente fuera del espacio en t={m.t}'
print('Invariantes OK: rapidez = 1 y posiciones dentro del espacio en los', m.t, 'pasos')

# Casos límite
def pol_final(**cambios):
    m = BoidsModel(dict(parameters2D, **cambios))
    m.run(display=False)
    return polarizacion(m)

print(f"Un solo agente corre sin errores, polarización = {pol_final(population=1):.2f} (trivialmente 1)")
print(f"Sin interacción (cohesión = separación = alineación = 0): {pol_final(cohesion_strength=0, seperation_strength=0, alignment_strength=0):.3f}")
print(f"Alineación alta (alignment_strength = 1): {pol_final(alignment_strength=1):.3f}")

## 2. Parte II: mejoras candidatas y priorización

Las limitaciones del modelo base (sección 1, pregunta 9c) sugieren mejoras de cuatro tipos, según la parte del sistema que modifican:
- **Sensores:** qué percibe el ave.
- **Decisión:** cómo elige qué hacer con lo que percibe.
- **Estado interno y actuadores:** qué acumula el ave y qué tan capaz es de ejecutar su decisión.
- **Ambiente:** el espacio y lo que hay en él.

**Criterios de priorización**
- **Factibilidad** en el tiempo disponible.
- **Definición clara:** un mecanismo concreto con parámetros controlables.
- **Medible:** produce un indicador que lo distingue del modelo base.
- **Respaldo:** datos o literatura que justifiquen el mecanismo y sus valores.
- **Orden:** primero los cambios en el agente y después los del ambiente, para no tener que volver a modificar al agente al agregar elementos externos.

### 2.1 Mejoras candidatas

Cada candidata es un mecanismo mínimo. La factibilidad va de 1 (difícil) a 5 (sencilla).

| # | Mejora | Tipo | Criterios | Respaldo | Factibilidad | Decisión |
|---|---|---|---|---|---|---|
| 1 | Escala física (metros y segundos) | Ambiente | C2, C5, C6 | Rapidez de parvadas y tiempo de reacción del estornino | 5 | Etapa 0 |
| 2 | Giro máximo por paso | Actuador | C3 | Carga máxima en giros (Usherwood et al., 2011) | 5 | Etapa 0 |
| 3 | Espacio sin paredes (torus) | Ambiente | C2 | El cielo no tiene paredes; en el modelo base los bordes afectan al 64% del área | 4 | Etapa 0 |
| 4 | Radio de percepción ajustable | Sensor | C3 | Parámetro de control para los experimentos | 5 | Mejora 1 |
| 5 | Vecinos topológicos (los *k* más cercanos) | Sensor | C3, C6 | Los estorninos siguen a 6 o 7 vecinos (Ballerini et al., 2008a) | 4 | Mejora 1 |
| 6 | Campo visual con punto ciego trasero | Sensor | C3 | Punto ciego trasero amplio en el estornino (Tisdale y Fernández-Juricic, 2009) | 4 | Mejora 1 |
| 7 | Ruido en la percepción | Sensor | C4 | Supuesto | 4 | Futuro |
| 8 | Líderes informados con un destino | Decisión | C1, C3 | Una minoría informada basta para guiar al grupo (Couzin et al., 2005) | 4 | Mejora 2 |
| 9 | Proporción de cada rol controlable | Decisión | C1, C8 | Permite barrer la fracción de líderes y de juveniles | 5 | Mejora 2 |
| 10 | Parámetros propios de cada rol | Decisión | C1 | Cualquier regla puede tener un valor distinto por rol | 5 | Mejora 2 |
| 11 | Juveniles con mayor gasto de vuelo | Estado | C1 | Los juveniles gastan ~14% más en vuelo (Rotics et al., 2016) | 4 | Mejora 2 y 3 |
| 12 | Retraso de reacción en juveniles | Decisión | C1 | No se encontró evidencia | — | Descartada |
| 13 | Gasto de energía según la rapidez | Estado | C3, C6 | Potencia de vuelo del estornino (Ward et al., 2001) | 4 | Mejora 3 |
| 14 | Gasto de energía en los giros | Estado | C3 | Duplicar la carga en un giro cuadruplica el esfuerzo (Usherwood et al., 2011); girar es costoso (Wilson et al., 2013) | 4 | Mejora 3 |
| 15 | Gasto creciente por giros consecutivos, con tope | Estado | C3 | Supuesto razonado | 3 | Mejora 3 |
| 16 | Recuperación de energía distinta en cada ave, con umbral | Estado | C1, C4 | La tasa metabólica varía de forma consistente entre individuos (Versteegh et al., 2008) | 4 | Mejora 3 |
| 17 | Rapidez máxima que depende de la energía | Actuador | C3 | Rango de rapidez observado (Ballerini et al., 2008b) | 4 | Mejora 3 |
| 18 | Masa corporal | Estado | C1 | Su efecto principal ya se representa con el gasto de energía | — | Descartada |
| 19 | Edad | Estado | C1 | Sin un mecanismo propio; su efecto (experiencia) ya se representa con los roles | — | Descartada |
| 20 | Actualización sincrónica de velocidades | Decisión | C5 | El orden de activación tiene un efecto agregado pequeño (sección 1, pregunta 5) | 5 | Futuro |
| 21 | Separación que crece al acercarse | Decisión | C3 | Limitación del modelo base, compensada con la calibración (sección 4.1) | 4 | Futuro |
| 22 | Viento variable en el espacio y el tiempo | Ambiente | C2, C4 | Un viento uniforme no cambia la estructura de la parvada; uno variable sí | 3 | Futuro |
| 23 | Obstáculos | Ambiente | C2 | — | 3 | Futuro |
| 24 | Depredador | Ambiente | C1, C2 | Requiere un segundo tipo de agente | 2 | Futuro |
| 25 | Espacio en 3D | Ambiente | C2 | Los datos de referencia son 3D; aumenta el costo de cómputo | 3 | Futuro |

### 2.2 Mejoras elegidas

**Etapa 0, modelo de referencia (candidatas 1 a 3).** No es una mejora de comportamiento, sino el requisito para comparar con datos reales: da escala física al modelo y corrige lo que no es físicamente posible.

1. **Percepción realista (candidatas 4 a 6).** Cada ave responde solo a sus *k* vecinos más cercanos, dentro de un radio y fuera de su punto ciego.
2. **Roles (candidatas 8 a 11).** Las aves se dividen en líderes informados, adultos y juveniles, en proporciones controlables y con parámetros propios por rol. Los líderes tienen un destino que se renueva cada vez que lo alcanzan.
3. **Energía (candidatas 13 a 17 y 11).** Volar rápido y girar gastan energía, y los giros seguidos gastan cada vez más, hasta un tope. Cada ave recupera energía a su propio ritmo cuando su gasto queda por debajo de ese ritmo. Con poca energía, baja su rapidez máxima.

**Cómo se complementan.** Las tres responden una misma pregunta: **¿cómo se mantiene unida una parvada cuyos miembros difieren en lo que perciben, lo que saben y lo que pueden sostener?** La percepción define a quién responde cada ave; los roles introducen información desigual (líderes) y capacidad desigual (juveniles); y la energía convierte esa diferencia de capacidad en aves rezagadas. Con eso se pueden estudiar dos preguntas concretas:
- ¿Cuántos líderes bastan para guiar a la parvada?
- Cuando algunas aves se cansan, ¿el grupo se adapta a ellas o las deja atrás?

## 3. Parte III: modelo extendido

Las mejoras se construyen por etapas. Cada etapa es una clase que hereda de la anterior y agrega un solo mecanismo. Con sus parámetros nuevos en el valor por defecto, cada etapa se comporta igual que la anterior; así se puede medir el efecto de cada mejora por separado (sección 6).

| Etapa | Sección | Qué agrega |
|---|---|---|
| 0 | 3.0 | Escala realista y espacio sin paredes (modelo de referencia) |
| 1 | 3.1 | Percepción |
| 2 | 3.2 | Roles |
| 3 | 3.3 | Energía |

En las tablas de parámetros, la columna **Fuente** indica si el valor es un **dato** medido (con su cita), un valor **derivado** de datos, un valor **calibrado** (sección 4) o un **supuesto** razonado.

### 3.0 Modelo de referencia: escala realista

El modelo base no tiene unidades: el espacio mide 50 × 50 "unidades" y un paso es un instante sin duración. Para poder compararlo con aves reales, esta etapa fija una escala física y corrige los aspectos del modelo base que no son físicamente posibles. No agrega comportamientos nuevos: es la **referencia** contra la que se comparan las mejoras.

**Escala.** 1 unidad = 1 m y 1 paso = 0,1 s.
- Con esta escala, la rapidez del modelo base (1 unidad por paso) equivale a **10 m/s**. Coincide con la rapidez de parvadas reales de estornino, que en 10 parvadas medidas en 3D va de 6,9 a 15,2 m/s, con media de 10,6 m/s (Ballerini et al., 2008b).
- 0,1 s es del orden del tiempo de reacción del estornino, 76 ms (Pomeroy y Heppner, 1977). Cada paso corresponde aproximadamente a una decisión del ave.

**Cambios respecto al modelo base**
1. **Rumbo y rapidez separados.** La velocidad se descompone en un rumbo unitario (`heading`) y una rapidez (`speed`). En esta etapa la rapidez sigue siendo constante; la Etapa 3 la hace variable.
2. **Giro máximo por paso.** En el modelo base un agente puede girar cualquier ángulo en un paso. Un ave gira inclinándose, y la fuerza que soporta en el giro es limitada: las palomas alcanzan hasta 2 g en giros con aleteo (Usherwood et al., 2011). Con factor de carga *n*, la aceleración lateral máxima es *a* = *g*·√(*n*² − 1), y el giro máximo por paso es Δθ = *a*·Δ*t* / *v*. Con *n* = 2, el giro máximo es de ≈ 9,7° por paso a 10 m/s. No se encontró un valor publicado para estornino, así que el de paloma se usa como aproximación.
3. **Espacio sin paredes.** El cielo no tiene paredes, y en el modelo base la regla de bordes afecta al 64% del área (sección 1, pregunta 3). Aquí el espacio es un **torus**: lo que sale por un lado entra por el opuesto. Las reglas usan el desplazamiento más corto entre dos aves, que puede cruzar el borde, y la regla de bordes desaparece.
4. **Posición inicial compacta.** Las aves empiezan en un parche cuadrado en el centro, con distancia media al vecino de 1 m y rumbos aleatorios. Así la densidad inicial no depende del tamaño del espacio.

**Tamaño del torus.** Debe ser lo bastante grande para que la parvada no se alcance a sí misma por el otro lado: la extensión de la parvada más dos radios de percepción debe ser menor que el lado. Con 200 aves y ~1 m entre vecinas, una parvada compacta ocupa unos 24 × 8 m (área ≈ *N*·*d*², con relación de aspecto ~3, dentro del rango 1,9–3,6 medido por Ballerini et al., 2008b). Se usa un lado de 200 m; la sección 4.1 verifica que la extensión observada cumple la condición.

#### Utilidades geométricas

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

G = 9.81  # Aceleración de la gravedad (m/s²)


def desplazamiento_toroidal(origen, destino, size):
    """ Vector más corto de `origen` a `destino` en un torus cuadrado de lado `size`.
    Funciona con puntos sueltos o con arreglos de puntos (broadcasting de numpy). """
    d = np.asarray(destino) - np.asarray(origen)
    return d - size * np.round(d / size)


def rotar_hacia(heading, deseada, max_angulo):
    """ Gira el vector unitario `heading` hacia la dirección `deseada` (2D),
    como máximo `max_angulo` radianes. Devuelve el nuevo rumbo y el ángulo girado. """
    angulo = np.arctan2(heading[0] * deseada[1] - heading[1] * deseada[0],
                        heading @ deseada)
    giro = np.clip(angulo, -max_angulo, max_angulo)
    c, s = np.cos(giro), np.sin(giro)
    nuevo = np.array([c * heading[0] - s * heading[1],
                      s * heading[0] + c * heading[1]])
    return nuevo, giro

#### Agente

In [ ]:
class BoidReferencia(ap.Agent):
    """ Boid con escala física: 1 unidad = 1 m y 1 paso = dt segundos.

    Diferencias con el modelo base:
    - Dirección (`heading`) y rapidez (`speed`) separadas; `velocity` = heading × speed.
    - Giro máximo por paso, limitado por el factor de carga máximo del vuelo.
    - Vecindad en un torus: las reglas usan el desplazamiento más corto.
    - Sin regla de bordes, porque el torus no tiene paredes. """

    def setup(self):
        self.heading = normalize(self.model.nprandom.random(self.p.ndim) - 0.5)
        self.speed = self.p.speed_pref
        self.velocity = self.heading * self.speed
        self.turn = 0.0  # Ángulo girado en el último paso (rad)

    def setup_pos(self, space):
        self.space = space
        self.pos = space.positions[self]

    def max_turn(self):
        """ Giro máximo por paso (rad) a la rapidez actual.
        Aceleración lateral máxima a = g·√(n² − 1); giro Δθ = a·dt / v,
        con v = speed / dt la rapidez en m/s. """
        a_max = G * np.sqrt(self.p.max_load_factor ** 2 - 1)
        return a_max * self.p.dt ** 2 / self.speed

    def social_force(self):
        """ Suma de las reglas de cohesión, separación y alineación. """
        nbs = self.space.neighbors(self, distance=self.p.outer_radius)
        force = np.zeros(self.p.ndim)
        if len(nbs) == 0:
            return force
        disp = desplazamiento_toroidal(self.pos, np.array(nbs.pos), self.p.size)
        dist = np.linalg.norm(disp, axis=1)

        # Regla 1 - Cohesión: hacia el centro de los vecinos
        force += disp.mean(axis=0) * self.p.cohesion_strength
        # Regla 2 - Separación: alejarse de los vecinos dentro de inner_radius
        force -= disp[dist < self.p.inner_radius].sum(axis=0) * self.p.separation_strength
        # Regla 3 - Alineación: igualar la velocidad promedio de los vecinos
        force += (np.mean(np.array(nbs.velocity), axis=0) - self.velocity) * self.p.alignment_strength
        return force

    def update_velocity(self):
        deseada = self.velocity + self.social_force()
        self.heading, self.turn = rotar_hacia(self.heading, deseada, self.max_turn())
        self.velocity = self.heading * self.speed

    def update_position(self):
        self.space.move_to(self, np.mod(self.pos + self.velocity, self.p.size))

#### Modelo

In [ ]:
class ModeloReferencia(ap.Model):
    """ Modelo de boids con escala realista en un torus. Es la referencia
    contra la que se comparan las mejoras. """

    agent_class = BoidReferencia

    def setup(self):
        assert self.p.ndim == 2, 'El modelo de referencia es bidimensional'
        self.space = ap.Space(self, shape=[self.p.size] * 2, torus=True)
        self.agents = ap.AgentList(self, self.p.population, self.agent_class)
        self.space.add_agents(self.agents, positions=self.posiciones_iniciales())
        self.agents.setup_pos(self.space)

    def posiciones_iniciales(self):
        """ Parche cuadrado en el centro del espacio. Para puntos uniformes con
        densidad ρ, la distancia media al vecino más cercano es ≈ 0.5/√ρ; el lado
        del parche se elige para que esa distancia sea `init_nnd`. """
        lado = 2 * self.p.init_nnd * np.sqrt(self.p.population)
        centro = self.p.size / 2
        return [centro + (self.nprandom.random(2) - 0.5) * lado
                for _ in range(self.p.population)]

    def step(self):
        self.agents.update_velocity()
        self.agents.update_position()

#### Métricas de la parvada

Se calculan sobre las distancias del torus:
- **Polarización:** magnitud del rumbo promedio (0 = desorden, 1 = todos en la misma dirección).
- **Distancia al vecino:** promedio de la distancia de cada ave a su vecina más cercana.
- **Fracción en colisión:** aves con una vecina a menos de 0,38 m, la distancia mínima observada entre estorninos (Ballerini et al., 2008b).
- **Grupos:** dos aves pertenecen al mismo grupo si están conectadas por una cadena de vecinas a menos de `outer_radius`. Se reporta el número de grupos y la fracción de aves en el mayor.
- **Extensión:** mayor distancia entre dos aves del grupo mayor.

In [ ]:
def distancias_toroidales(m):
    """ Matriz de distancias entre todos los agentes, por el camino más corto en el torus. """
    pos = np.array(list(m.space.positions.values()))
    return np.linalg.norm(desplazamiento_toroidal(pos[:, None], pos[None], m.p.size), axis=-1)


def metricas_parvada(m):
    """ Indicadores del estado de la parvada en el paso actual. """
    d = distancias_toroidales(m)
    np.fill_diagonal(d, np.inf)
    nnd = d.min(axis=1)
    n_grupos, etiquetas = connected_components(csr_matrix(d < m.p.outer_radius), directed=False)
    tamanos = np.bincount(etiquetas)
    mayor = etiquetas == tamanos.argmax()
    d_mayor = d[np.ix_(mayor, mayor)]
    extension = d_mayor[np.isfinite(d_mayor)].max() if mayor.sum() > 1 else 0.0
    return {
        'polarizacion': np.linalg.norm(np.mean([a.heading for a in m.agents], axis=0)),
        'dist_vecino': nnd.mean(),
        'frac_colision': (nnd < m.p.collision_distance).mean(),
        'n_grupos': n_grupos,
        'frac_grupo_mayor': tamanos.max() / len(etiquetas),
        'extension': extension,
    }


class ModeloReferenciaConDatos(ModeloReferencia):
    """ Modelo de referencia que registra las métricas en cada paso. """

    def update(self):
        for k, v in metricas_parvada(self).items():
            self.record(k, v)

#### Parámetros

| Parámetro | Valor | Unidad | Fuente |
|---|---|---|---|
| `dt` | 0,1 | s | Dato: tiempo de reacción de 76 ms (Pomeroy y Heppner, 1977) |
| `speed_pref` | 1,0 | m/paso (10 m/s) | Dato: rapidez de parvadas, 6,9–15,2 m/s (Ballerini et al., 2008b) |
| `max_load_factor` | 2 | *g* | Dato en paloma (Usherwood et al., 2011), usado como aproximación |
| `size` | 200 | m | Derivado; verificado en la sección 4.1 |
| `population` | 200 | aves | Supuesto, por costo de cómputo; las parvadas medidas tienen 448–2631 aves |
| `init_nnd` | 1,0 | m | Dato: distancia al vecino observada (Ballerini et al., 2008b) |
| `collision_distance` | 0,38 | m | Dato: distancia mínima entre aves (Ballerini et al., 2008b) |
| `inner_radius` | 1,0 | m | Calibrado (sección 4.1) |
| `separation_strength` | 0,4 | — | Calibrado (sección 4.1) |
| `outer_radius` | 10 | m | Modelo base |
| `cohesion_strength` | 0,005 | — | Modelo base |
| `alignment_strength` | 0,3 | — | Modelo base |

In [ ]:
parametros_ref = {
    # Simulación
    'seed': 123,
    'steps': 1000,             # 1000 pasos = 100 s de vuelo
    'ndim': 2,
    'population': 200,
    # Escala
    'dt': 0.1,                 # s por paso
    'size': 200,               # m, lado del torus
    'init_nnd': 1.0,           # m, distancia al vecino en la posición inicial
    'collision_distance': 0.38,  # m, distancia mínima observada entre aves
    # Vuelo
    'speed_pref': 1.0,         # m/paso (= 10 m/s)
    'max_load_factor': 2.0,    # g
    # Reglas
    'inner_radius': 1.0,         # m, calibrado en la sección 4.1
    'outer_radius': 10,        # m
    'cohesion_strength': 0.005,
    'separation_strength': 0.4,  # calibrado en la sección 4.1
    'alignment_strength': 0.3,
}

#### Simulación de referencia

Una corrida de 1000 pasos (100 s de vuelo).

In [ ]:
ref = ModeloReferenciaConDatos(parametros_ref)
salida_ref = ref.run(display=False)
serie_ref = salida_ref.variables.ModeloReferenciaConDatos

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
serie_ref['polarizacion'].plot(ax=axs[0], title='Polarización')
serie_ref['dist_vecino'].plot(ax=axs[1], title='Distancia al vecino (m)')
axs[1].axhspan(0.68, 1.51, color='green', alpha=0.15, label='Rango observado')
axs[1].legend()
pos = np.array(list(ref.space.positions.values()))
rumbos = np.array([a.heading for a in ref.agents])
axs[2].quiver(pos[:, 0], pos[:, 1], rumbos[:, 0], rumbos[:, 1], angles='xy', scale=40, width=0.003)
axs[2].set(xlim=(0, ref.p.size), ylim=(0, ref.p.size), aspect='equal', title=f't = {ref.t} (posición y rumbo)')
for ax in axs[:2]:
    ax.set_xlabel('Paso t')
plt.tight_layout()
plt.show()
serie_ref.loc[[0, 100, 250, 500, 1000]].round(3)

### 3.1 Mejora 1: (nombre)

**Qué cambia:**

**Justificación (literatura / supuesto razonado):**

**Parámetros nuevos:** (nombre, valor por defecto, rango, fuente)

In [ ]:
# TODO: implementación de la mejora 1

### 3.2 Mejora 2: (nombre)

**Qué cambia:**

**Justificación (literatura / supuesto razonado):**

**Parámetros nuevos:** (nombre, valor por defecto, rango, fuente)

In [ ]:
# TODO: implementación de la mejora 2

### 3.3 Mejora 3: (nombre)

**Qué cambia:**

**Justificación (literatura / supuesto razonado):**

**Parámetros nuevos:** (nombre, valor por defecto, rango, fuente)

In [ ]:
# TODO: implementación de la mejora 3

### 3.4 Modelo extendido completo

In [ ]:
# TODO: clases finales (agente y modelo) con las tres mejoras integradas, con docstrings

In [ ]:
# TODO: corrida de demostración y figura / animación

## 4. Calibración

### 4.1 Calibración del modelo de referencia

Con los valores originales del modelo base (`inner_radius` = 3, separación 0,1) pero en el espacio sin paredes, la distancia al vecino queda en ~3,1 m, el doble de lo observado. En la Parte I la distancia era ~1 m porque las paredes **comprimían** a la parvada, no por sus reglas.

Se calibran dos parámetros contra dos datos de 10 parvadas de estornino (Ballerini et al., 2008b):
- **Distancia media al vecino:** entre 0,68 y 1,51 m, con mediana de 1,05 m.
- **Distancia mínima:** casi ninguna ave a menos de 0,38 m (zona de exclusión de 0,19 m de radio, comparable a la envergadura de 0,4 m).

**Procedimiento:** 20 réplicas por valor, 1000 pasos cada una, y métricas promediadas entre los pasos 500 y 1000, cuando la parvada ya se formó.
1. Barrido de `inner_radius` con la separación original (0,1).
2. Con `inner_radius` fijo, barrido de `separation_strength`.

In [ ]:
def calibrar(cambios):
    """ Corre 20 réplicas por configuración y promedia las métricas de los pasos 500 a 1000. """
    muestra = ap.Sample(dict(parametros_ref, **cambios))
    exp = ap.Experiment(ModeloReferenciaConDatos, muestra, iterations=20, record=True, randomize=True)
    variables = exp.run(n_jobs=-1, verbose=0).arrange_variables()
    barrido = [k for k, v in cambios.items() if isinstance(v, ap.Values)][0]
    por_corrida = (variables[variables.t >= 500]
                   .groupby([barrido, 'iteration'])
                   [['dist_vecino', 'frac_colision', 'polarizacion', 'frac_grupo_mayor', 'n_grupos']]
                   .mean())
    return variables, por_corrida

var_r, cal_r = calibrar({'inner_radius': ap.Values(0.5, 0.75, 1.0, 1.5, 2.0, 3.0),
                          'separation_strength': 0.1})
cal_r.groupby('inner_radius').agg(['mean', 'std']).round(3)

In [ ]:
var_s, cal_s = calibrar({'inner_radius': 1.0,
                          'separation_strength': ap.Values(0.1, 0.2, 0.4, 0.8)})
cal_s.groupby('separation_strength').agg(['mean', 'std']).round(3)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, cal, x, titulo in [(axs[0], cal_r, 'inner_radius', 'Paso 1: inner_radius (separación = 0,1)'),
                           (axs[1], cal_s, 'separation_strength', 'Paso 2: separación (inner_radius = 1 m)')]:
    resumen = cal.groupby(x).agg(['mean', 'std'])
    ax.errorbar(resumen.index, resumen[('dist_vecino', 'mean')], yerr=resumen[('dist_vecino', 'std')],
                marker='o', capsize=3, label='Distancia al vecino (m)')
    ax.axhspan(0.68, 1.51, color='green', alpha=0.15, label='Rango observado (0,68–1,51 m)')
    ax2 = ax.twinx()
    ax2.plot(resumen.index, 100 * resumen[('frac_colision', 'mean')], 'r--s', label='Aves en colisión (%)')
    ax2.set_ylabel('Aves a menos de 0,38 m (%)', color='r')
    ax.set(xlabel=x, ylabel='Distancia al vecino (m)', title=titulo)
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

elegida = var_s[var_s.separation_strength == parametros_ref['separation_strength']]
limite = parametros_ref['size'] - 2 * parametros_ref['outer_radius']
print(f"Extensión máxima del grupo mayor en las 20 réplicas: {elegida.extension.max():.1f} m "
      f"(límite: {limite} m) → {'cumple' if elegida.extension.max() < limite else 'NO cumple'}")

**Resultados**

- **Paso 1.** La distancia al vecino crece casi en proporción a `inner_radius`: es ~0,6 m con 0,5 m y ~3,1 m con 3 m. Con `inner_radius` = 1 m queda en 1,01 m, junto a la mediana observada. Sin embargo, entre 3% y 5% de las aves queda a menos de 0,38 m de otra. Esto viene de una limitación del modelo base (sección 1, pregunta 4c): la separación empuja más cuanto **más lejos** está el vecino, así que es débil justo cuando dos aves están muy cerca.
- **Paso 2.** Con separación de 0,2 a 0,4, la fracción en colisión baja a 0,1–0,3% y la distancia al vecino sube a ~1,13 m, todavía dentro del rango observado. Con 0,8 la polarización empieza a bajar.

**Valores elegidos:** `inner_radius` = 1 m y `separation_strength` = 0,4. Con ellos, la distancia al vecino es ~1,13 m, la fracción en colisión ~0,1% y la polarización ~0,84.

**Tamaño del torus:** la extensión máxima del grupo mayor queda por debajo del límite de 180 m (200 m menos dos radios de percepción), así que la parvada no se alcanza a sí misma.

**Observación para la validación:** en promedio, la parvada se divide en 2 o 3 grupos y el mayor reúne ~73% de las aves.

### 4.2 Análisis de sensibilidad

In [ ]:
# TODO: análisis de sensibilidad de los parámetros clave (±20%, uno a la vez)

## 5. Recolección de datos (KPIs)

Cada KPI debe ligarse a un mecanismo del modelo. Candidatos: polarización, distancia media al vecino más cercano, dispersión o cohesión del grupo, número de colisiones, supervivencia ante depredador, energía media.

| KPI | Definición | Mecanismo que mide |
|-----|------------|--------------------|
|     |            |                    |

In [ ]:
# TODO: record() en el modelo, y función que extrae/limpia el DataDict

## 6. Experimentos

- Barrido de parámetros con `ap.Experiment` / `ap.Sample` / `ap.Range`.
- **≥ 20 réplicas** por configuración, semillas distintas.
- Chequeos de robustez (variar tamaño de la población, tamaño del espacio, etc.).

In [ ]:
# TODO: definición del experimento y ejecución

In [ ]:
# TODO: figuras con media e intervalo de confianza entre réplicas; tablas resumen

## 7. Validación

- **Pruebas de código:** casos límite (p. ej. un solo agente, cohesión = 0, sin depredador) con resultado esperado.
- **Validez aparente:** ¿el comportamiento se parece visualmente y cualitativamente a una bandada?
- **Comparación con datos externos o hechos estilizados:** (fuente, estadístico, resultado).

In [ ]:
# TODO: pruebas y comparaciones

## 8. Conclusiones y limitaciones

- Hallazgos principales.
- Qué quedó fuera y por qué.
- Cómo se podría extender.

## 9. Referencias

- Ballerini, M., Cabibbo, N., Candelier, R., Cavagna, A., Cisbani, E., Giardina, I., Lecomte, V., Orlandi, A., Parisi, G., Procaccini, A., Viale, M. y Zdravkovic, V. (2008a). Interaction ruling animal collective behavior depends on topological rather than metric distance: evidence from a field study. *Proceedings of the National Academy of Sciences*, 105(4), 1232–1237. https://doi.org/10.1073/pnas.0711437105
- Ballerini, M., Cabibbo, N., Candelier, R., Cavagna, A., Cisbani, E., Giardina, I., Orlandi, A., Parisi, G., Procaccini, A., Viale, M. y Zdravkovic, V. (2008b). Empirical investigation of starling flocks: a benchmark study in collective animal behaviour. *Animal Behaviour*, 76(1), 201–215. https://arxiv.org/abs/0802.1667
- Couzin, I. D., Krause, J., Franks, N. R. y Levin, S. A. (2005). Effective leadership and decision-making in animal groups on the move. *Nature*, 433, 513–516. https://doi.org/10.1038/nature03236
- Foramitti, J. (2021). AgentPy: A package for agent-based modeling in Python. *Journal of Open Source Software*, 6(62), 3065. https://doi.org/10.21105/joss.03065
- Pomeroy, H. y Heppner, F. (1977). Laboratory determination of startle reaction time of the starling (*Sturnus vulgaris*). *Animal Behaviour*, 25, 720–725. https://doi.org/10.1016/0003-3472(77)90121-X
- Reynolds, C. W. (1987). Flocks, herds and schools: A distributed behavioral model. *ACM SIGGRAPH Computer Graphics*, 21(4), 25–34. https://doi.org/10.1145/37402.37406
- Rotics, S., Kaatz, M., Resheff, Y. S., Turjeman, S. F., Zurell, D., Sapir, N., Eggers, U., Flack, A., Fiedler, W., Jeltsch, F., Wikelski, M. y Nathan, R. (2016). The challenges of the first migration: movement and behaviour of juvenile vs. adult white storks with insights regarding juvenile mortality. *Journal of Animal Ecology*, 85(4), 938–947. https://doi.org/10.1111/1365-2656.12525
- Tisdale, V. y Fernández-Juricic, E. (2009). Vigilance and predator detection vary between avian species with different visual acuity and coverage. *Behavioral Ecology*, 20(5), 936–945. https://ideas.repec.org/a/oup/beheco/v20y2009i5p936-945.html
- Usherwood, J. R., Stavrou, M., Lowe, J. C., Roskilly, K. y Wilson, A. M. (2011). Flying in a flock comes at a cost in pigeons. *Nature*, 474, 494–497. https://doi.org/10.1038/nature10164
- Versteegh, M. A., Helm, B., Dingemanse, N. J. y Tieleman, B. I. (2008). Repeatability and individual correlates of basal metabolic rate and total evaporative water loss in birds: a case study in European stonechats. *Comparative Biochemistry and Physiology Part A*, 150(4), 452–457. https://doi.org/10.1016/j.cbpa.2008.05.006
- Ward, S., Möller, U., Rayner, J. M. V., Jackson, D. M., Bilo, D., Nachtigall, W. y Speakman, J. R. (2001). Metabolic power, mechanical power and efficiency during wind tunnel flight by the European starling *Sturnus vulgaris*. *Journal of Experimental Biology*, 204(19), 3311. https://cob.silverchair.com/jeb/article-pdf/204/19/3311/1238626/3311.pdf
- Wilson, R. P. et al. (2013). Turn costs change the value of animal search paths. *Ecology Letters*, 16(9), 1145–1150. https://doi.org/10.1111/ele.12149